# Stochastic convergence

## A visual guide to convergence in distribution, in probability, and almost surely

When the objects we study are deterministic (e.g. real numbers, functions, sequences), convergence has an unambiguous meaning. We specify a notion of distance, and we ask whether the terms of a sequence eventually stay within any prescribed tolerance of a limit.

In statistics, however, the objects we work with are random. An estimator like $\bar X_n$ does not produce a single number for each $n$ — it produces a different value each time we repeat the experiment. So when we write $\bar X_n \to \mu$, the deterministic definition no longer applies directly: there is no single sequence of numbers to check against the limit. Instead, for each sample size $n$, $\bar X_n$ has a distribution, and for each realisation of the data, $\bar X_n$ traces out a different path.

This means there are genuinely different ways to ask whether a sequence of random variables "converges." This notebook gives an intuitive explanation of three of them:

- convergence in distribution;
- convergence in probability;
- almost sure convergence.

All figures are saved into `content/blog/stochastic-convergence/` so the Hugo post can include them directly.

In [1]:
from __future__ import annotations

from math import erf, pi, sqrt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

RNG = np.random.default_rng(20260406)


def locate_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "content").exists() and (cwd / "notebooks").exists():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "content").exists():
        return cwd.parent
    for parent in [cwd, *cwd.parents]:
        if (parent / "content").exists() and (parent / "notebooks").exists():
            return parent
    raise RuntimeError("Could not locate repository root.")


ROOT = locate_root()
OUTDIR = ROOT / "content" / "blog" / "stochastic-convergence"
OUTDIR.mkdir(parents=True, exist_ok=True)


def normal_pdf(x: np.ndarray, mean: float = 0.0, sd: float = 1.0) -> np.ndarray:
    z = (x - mean) / sd
    return np.exp(-0.5 * z**2) / (sd * sqrt(2 * pi))


def normal_cdf(x: np.ndarray, mean: float = 0.0, sd: float = 1.0) -> np.ndarray:
    z = (x - mean) / (sd * sqrt(2.0))
    return 0.5 * (1.0 + np.vectorize(erf)(z))


def savefig(name: str) -> Path:
    path = OUTDIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight", dpi=180)
    plt.close()
    print(path.relative_to(ROOT))
    return path


ROOT, OUTDIR

(WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog'),
 WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog/content/blog/stochastic-convergence'))

## 1. Convergence for deterministic sequences

A sequence of real numbers $(x_n)$ converges to a limit $x$ if, for every $\varepsilon > 0$, there exists $N \in \mathbb{N}$ such that

$$
n \geq N \implies |x_n - x| < \varepsilon.
$$

The definition asks for something specific: after some finite index, *every* subsequent term lies within $\varepsilon$ of the limit, no matter how small $\varepsilon$ is chosen. The sequence does not merely visit the neighbourhood of $x$ — it enters and remains there permanently.

### Pointwise and uniform convergence for functions

Even in deterministic analysis, there is more than one notion of convergence. Consider a sequence of functions $f_n : [0,1] \to \mathbb{R}$.

**Pointwise convergence** says that for each fixed $x \in [0,1]$, the numerical sequence $f_n(x)$ converges. The rate at which $f_n(x)$ approaches its limit may depend on $x$, and the index $N$ required to achieve a given tolerance can vary across the domain.

**Uniform convergence** is stronger: it asks for a single $N$ that works simultaneously for all $x$:

$$
\forall\, \varepsilon > 0,\;\exists\, N \in \mathbb{N} \text{ such that } n \geq N \implies \sup_{x \in [0,1]} |f_n(x) - f(x)| < \varepsilon.
$$

The difference matters because pointwise convergence can fail to preserve continuity, integrability, and interchange of limits, which uniform convergence does preserve.



In [2]:
# Figure 1: deterministic sequence converging to a limit
n = np.arange(1, 31)
limit = 1.0
eps = 0.18
x_n = limit + 0.75 * np.exp(-n / 6.5) * np.cos(0.9 * n)
inside = np.abs(x_n - limit) < eps
N_trap = int(np.argmax(np.logical_and.accumulate(inside[::-1])[::-1]) + 1)

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.axhspan(limit - eps, limit + eps, color="#dbeafe", alpha=0.85,
           label=r"$x \pm \varepsilon$")
ax.axhline(limit, color="#1d4ed8", lw=2, label=r"limit $x$")
ax.plot(n, x_n, "o", color="#111827", ms=4, lw=0, zorder=3, label=r"$x_n$")
ax.axvline(N_trap, color="#dc2626", ls="--", lw=1.5, label=rf"$N = {N_trap}$")
ax.set(xlabel=r"$n$", ylabel=r"$x_n$")
ax.legend(frameon=False, loc="upper right", fontsize=9)
savefig("deterministic-sequence.png")

# Figure 2: pointwise vs uniform convergence of functions
x = np.linspace(0, 1, 500)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4), sharey=True)

cmap_left = plt.cm.Blues(np.linspace(0.35, 0.85, 5))
for i, m in enumerate([1, 2, 5, 15, 40]):
    axes[0].plot(x, x**m, lw=1.8, color=cmap_left[i])
axes[0].plot(x, np.where(x < 1, 0.0, 1.0), color="black", ls="--", lw=1.5)
axes[0].set_title(r"$f_n(x) = x^n$", fontsize=11)
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel(r"$f_n(x)$")
axes[0].set_ylim(-0.02, 1.08)

cmap_right = plt.cm.Oranges(np.linspace(0.35, 0.85, 5))
for i, m in enumerate([1, 2, 5, 15, 40]):
    axes[1].plot(x, x / (1 + m * x), lw=1.8, color=cmap_right[i])
axes[1].plot(x, np.zeros_like(x), color="black", ls="--", lw=1.5)
axes[1].set_title(r"$g_n(x) = x/(1+nx)$", fontsize=11)
axes[1].set_xlabel(r"$x$")
axes[1].set_ylabel(r"$g_n(x)$")

savefig("function-convergence.png")

content\blog\stochastic-convergence\deterministic-sequence.png


content\blog\stochastic-convergence\function-convergence.png


WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog/content/blog/stochastic-convergence/function-convergence.png')

## 2. Random variables as functions

Formally, a random variable is a measurable function $X : \Omega \to \mathbb{R}$, where $(\Omega, \mathcal{F}, \mathbb{P})$ is a probability space. The sample space $\Omega$ indexes all possible outcomes of an experiment; the randomness of $X$ comes entirely from the randomness of the outcome $\omega \in \Omega$. Once $\omega$ is determined, $X(\omega)$ is just a real number.

A sequence of random variables $(X_1, X_2, \ldots)$ is therefore a sequence of such functions, all defined on the same probability space. This gives rise to two fundamentally different ways of examining the sequence.

**Pathwise view.** Fix an outcome $\omega \in \Omega$. Then $X_1(\omega), X_2(\omega), \ldots$ is an ordinary numerical sequence — one specific realisation of the process. Different choices of $\omega$ produce different sequences.

**Distributional view.** Fix an index $n$. Then $X_n$ is a single random variable with its own distribution. As $n$ varies, we obtain a sequence of distributions.

These two views are why multiple notions of convergence arise. We can ask whether the *distributions* of $X_n$ approach a limiting distribution, whether the *probability* that $X_n$ deviates from a target vanishes, or whether the *realised sequences* $X_n(\omega)$ converge for almost every $\omega$. Each question leads to a different definition.

To make this concrete, consider $X_1, X_2, \ldots$ drawn i.i.d. from $N(0,1)$ and let $\bar X_n = \frac{1}{n}\sum_{i=1}^n X_i$. The next figure shows the same collection of realisations from two angles: on the left, sample paths of $\bar X_n(\omega)$ plotted against $n$; on the right, a histogram of $\bar X_n$ across realisations at a single fixed time.

In [3]:
# Figure 3: the same process seen pathwise and distributionally
n_max = 250
m_paths = 60
steps = RNG.normal(size=(m_paths, n_max))
running_means = np.cumsum(steps, axis=1) / np.arange(1, n_max + 1)
n_star = 180

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
for row in running_means[:35]:
    axes[0].plot(np.arange(1, n_max + 1), row, lw=1.0, alpha=0.55)
axes[0].axhline(0.0, color="black", ls="--", lw=1.4)
axes[0].set_xlabel(r"time $n$")
axes[0].set_ylabel(r"$\bar X_n(\omega)$")

vals = running_means[:, n_star - 1]
x_grid = np.linspace(vals.min() - 0.15, vals.max() + 0.15, 400)
axes[1].hist(vals, bins=14, density=True, color="#93c5fd", edgecolor="white")
axes[1].plot(x_grid, normal_pdf(x_grid, 0.0, 1 / np.sqrt(n_star)), color="#1d4ed8", lw=2)
axes[1].set_xlabel(r"value of $\bar X_{%d}$" % n_star)
axes[1].set_ylabel("density")

savefig("paths-vs-distribution.png")

content\blog\stochastic-convergence\paths-vs-distribution.png


WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog/content/blog/stochastic-convergence/paths-vs-distribution.png')

## 3. Convergence in distribution

Recall that the **distribution** (or **law**) of a random variable $X$ is the probability measure $P_X$ on $\mathbb{R}$ defined by $P_X(B) = \mathbb{P}(X \in B)$. It is completely determined by the **distribution function** $F_X(t) = \mathbb{P}(X \leq t)$.

We write $X_n \to_d X$ if the distribution functions converge pointwise at every continuity point of $F_X$:

$$
F_{X_n}(t) \to F_X(t) \quad \text{for all } t \text{ at which } F_X \text{ is continuous.}
$$

This is the weakest notion in the hierarchy. It is a statement purely about the sequence of distributions $P_{X_1}, P_{X_2}, \ldots$ and says nothing about whether the random variables $X_n$ and $X$ are close in any realisation. In particular, $X_n$ and $X$ need not even be defined on the same probability space.

When the limiting distribution is continuous, convergence in distribution implies $\mathbb{P}(X_n \in [a,b]) \to \mathbb{P}(X \in [a,b])$.

### The Central Limit Theorem

**Theorem (Central Limit Theorem).** *Let $X_1, \ldots, X_n$ be i.i.d. real-valued random variables with $\mathbb{E}(X_1) = \mu$ and $\text{Var}(X_1) = \sigma^2 < \infty$. Then*

$$
\sqrt{n}(\bar X_n - \mu) \to_d N(0, \sigma^2).
$$

The CLT does not say the centred-and-scaled statistic converges to a constant. It says its *distribution* approaches a Gaussian law. This is why convergence in distribution is the language of asymptotic normality, confidence intervals, and tests.

In [4]:
# Figure 4: laws changing shape toward a limit
x = np.linspace(-4, 4, 600)
fig, ax = plt.subplots(figsize=(8.3, 4.4))
for n_val, color in zip([1, 2, 5, 20], ["#93c5fd", "#60a5fa", "#2563eb", "#1e3a8a"]):
    sd = np.sqrt(1 + 1 / n_val)
    ax.plot(x, normal_pdf(x, 0.0, sd), lw=2.2, color=color, label=rf"$N(0, 1 + 1/{n_val})$")
ax.plot(x, normal_pdf(x, 0.0, 1.0), color="black", ls="--", lw=1.6, label=r"limit $N(0,1)$")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.legend(frameon=False)
savefig("distribution-shapes.png")

# Figure 5: a CLT simulation
replications = 5000
n_values = [2, 8, 32, 128]
fig, axes = plt.subplots(2, 2, figsize=(10.5, 7.5), sharex=True, sharey=True)
x_grid = np.linspace(-4, 4, 500)
for ax, n_val in zip(axes.ravel(), n_values):
    samples = RNG.exponential(scale=1.0, size=(replications, n_val))
    z = np.sqrt(n_val) * (samples.mean(axis=1) - 1.0)
    ax.hist(z, bins=35, density=True, color="#bfdbfe", edgecolor="white")
    ax.plot(x_grid, normal_pdf(x_grid, 0.0, 1.0), color="#1d4ed8", lw=2)
    ax.set_title(rf"$n={n_val}$")
    ax.axvline(0.0, color="black", lw=1, alpha=0.5)
for ax in axes[-1, :]:
    ax.set_xlabel(r"$\sqrt{n}(\bar X_n - 1)$")
for ax in axes[:, 0]:
    ax.set_ylabel("density")
savefig("clt-panel.png")

content\blog\stochastic-convergence\distribution-shapes.png


content\blog\stochastic-convergence\clt-panel.png


WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog/content/blog/stochastic-convergence/clt-panel.png')

## 4. Convergence in probability

Convergence in distribution only constrains the *shape* of the distribution of $X_n$. It does not require $X_n$ and $X$ to be defined on the same probability space, and says nothing about whether the values $X_n(\omega)$ are close to $X(\omega)$ for any particular outcome $\omega$. Convergence in probability fills this gap: it asks whether $X_n$ is close to $X$ as a random variable, not just in law. This requires both to live on the same $(\Omega, \mathcal{F}, \mathbb{P})$, because the expression $|X_n - X|$ must be well-defined.

We write $X_n \to_p X$ if for every $\varepsilon > 0$,

$$
\mathbb P(|X_n - X| > \varepsilon) \to 0.
$$

The definition is a statement about each fixed $n$ separately. For a given $n$, consider the set of outcomes where $X_n$ is more than $\varepsilon$ from $X$:

$$
A_n(\varepsilon) = \{\omega \in \Omega : |X_n(\omega) - X(\omega)| > \varepsilon\}.
$$

Convergence in probability says that for every $\varepsilon > 0$,

$$
\mathbb{P}(A_n(\varepsilon)) \to 0 \quad \text{as } n \to \infty.
$$

Note that the bad set $A_n(\varepsilon)$ is allowed to change with $n$: the outcomes $\omega$ in $A_n(\varepsilon)$, corresponding to the paths that fall outside the $\varepsilon$-band at time $n$, need not be the same across different times. All that matters is that the total probability of this set shrinks.

Geometrically, this has a clean interpretation. Suppose we simulate $M$ independent copies of the entire experiment, producing $M$ sample paths $X_1(\omega_j), X_2(\omega_j), \ldots$ for $j = 1, \ldots, M$. Plot all $M$ paths on a single figure, draw an $\varepsilon$-band around the target $X$, and then draw a vertical line at time $n$. The cross-section at that vertical line consists of $M$ points — one per path. The proportion of those points that fall outside the $\varepsilon$-band is

$$
\hat p_n = \frac{1}{M}\sum_{j=1}^{M} \mathbf{1}\{|X_n(\omega_j) - X(\omega_j)| > \varepsilon\},
$$

which, by the law of large numbers, approximates $\mathbb{P}(|X_n - X| > \varepsilon)$ for large $M$. Convergence in probability says $\hat p_n \to 0$ as $n \to \infty$.

### The Weak Law of Large Numbers

The WLLN is the canonical example: it states that the sample mean converges in probability to the population mean.

**Theorem (Weak Law of Large Numbers).** *Let $X_1, \ldots, X_n$ be i.i.d. real-valued random variables with $\text{Var}(X_1) < \infty$. Then*

$$
\bar X_n \to_p \mathbb{E}(X_1).
$$

The WLLN says that for large $n$, the sample mean is likely to be close to $\mu$: it is a per-$n$ guarantee that the probability of $\bar X_n$ being far from $\mu$ is small. It does not say that any particular realised sequence of running averages stays close permanently. The proof is a direct application of Markov's inequality (see appendix below).

Convergence in probability is a **per-snapshot** property. At each fixed time $n$, the probability mass concentrates inside the $\varepsilon$-band: the density of $X_n$ piles up near $X$, and the tails thin out. But this is a statement about each time-slice separately — it says nothing about what happens along a single path over time. A specific path could leave the band at time $n_1$, return, leave again at $n_2$, and so on. As long as the mass outside the band at any single time goes to zero, convergence in probability is satisfied. Whether individual paths eventually settle down permanently is a strictly stronger requirement, addressed in the next section.

In [5]:
# Figure 6: vertical-slice intuition for convergence in probability
n_max = 400
m_paths = 80
eps = 0.12
x = RNG.normal(size=(m_paths, n_max))
y = np.cumsum(x, axis=1) / np.arange(1, n_max + 1)
n_slice = 220
outside = np.abs(y[:, n_slice - 1]) > eps
p_hat = (np.abs(y) > eps).mean(axis=0)

fig, axes = plt.subplots(2, 1, figsize=(9.3, 8.5), sharex=True, gridspec_kw={"height_ratios": [4, 1.5]})
for idx, row in enumerate(y):
    color = "#dc2626" if outside[idx] else "#2563eb"
    alpha = 0.8 if outside[idx] else 0.18
    lw = 1.4 if outside[idx] else 1.0
    axes[0].plot(np.arange(1, n_max + 1), row, color=color, alpha=alpha, lw=lw)
axes[0].axhspan(-eps, eps, color="#dbeafe", alpha=0.85)
axes[0].axvline(n_slice, color="black", ls="--", lw=1.5)
axes[0].set_ylabel(r"$\bar X_n(\omega)$")
axes[0].text(n_slice + 6, eps + 0.035, rf"proportion outside at $n={n_slice}$: {outside.mean():.2f}", color="black")

axes[1].plot(np.arange(1, n_max + 1), p_hat, color="#1d4ed8", lw=2)
axes[1].scatter([n_slice], [p_hat[n_slice - 1]], color="#dc2626", zorder=3)
axes[1].set_xlabel(r"time $n$")
axes[1].set_ylabel(r"$\hat p_n$")
axes[1].set_ylim(-0.02, 1.02)
savefig("convergence-in-probability.png")

# Figure 7: WLLN spaghetti plot with outside paths in red
n_max = 300
m_paths = 50
eps_wlln = 0.05
n_slice_wlln = 200
bernoulli = RNG.binomial(1, 0.5, size=(m_paths, n_max))
running = np.cumsum(bernoulli, axis=1) / np.arange(1, n_max + 1)
outside_wlln = np.abs(running[:, n_slice_wlln - 1] - 0.5) > eps_wlln

fig, ax = plt.subplots(figsize=(8.6, 4.6))
for idx, row in enumerate(running):
    color = "#dc2626" if outside_wlln[idx] else "#2563eb"
    alpha = 0.8 if outside_wlln[idx] else 0.35
    lw = 1.4 if outside_wlln[idx] else 1.0
    ax.plot(np.arange(1, n_max + 1), row, lw=lw, alpha=alpha, color=color)
ax.axhspan(0.5 - eps_wlln, 0.5 + eps_wlln, color="#dbeafe", alpha=0.85)
ax.axhline(0.5, color="black", ls="--", lw=1.4)
ax.axvline(n_slice_wlln, color="black", ls="--", lw=1.5)
ax.set_xlabel(r"time $n$")
ax.set_ylabel(r"running mean $\bar X_n$")
savefig("wlln-spaghetti.png")

content\blog\stochastic-convergence\convergence-in-probability.png


content\blog\stochastic-convergence\wlln-spaghetti.png


WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog/content/blog/stochastic-convergence/wlln-spaghetti.png')

## 5. Almost sure convergence

Convergence in probability asks: at time $n$, is most of the probability mass near $X$? Almost sure convergence asks something much stronger: does each individual path $X_n(\omega)$ eventually settle down to $X(\omega)$?

We write $X_n \to_{a.s.} X$ if

$$
\mathbb P\bigl(\{\omega : X_n(\omega) \to X(\omega)\}\bigr) = 1.
$$

To unpack this, consider the set

$$
\mathcal{G} = \{\omega \in \Omega : X_n(\omega) \to X(\omega)\}.
$$

This is the set of all outcomes $\omega$ for which the realised sequence $X_1(\omega), X_2(\omega), \ldots$ converges to $X(\omega)$ in the ordinary deterministic sense. Its complement $\mathcal{N} = \Omega \setminus \mathcal{G}$ is the set of "bad" outcomes — those where the path fails to converge. Almost sure convergence says that $\mathcal{G}$ has full measure: $\mathbb{P}(\mathcal{G}) = 1$, or equivalently $\mathbb{P}(\mathcal{N}) = 0$. The bad set need not be empty, but it must be negligible under $\mathbb{P}$.

What does convergence of a single path actually require? For a fixed $\omega \in \mathcal{G}$ and any $\varepsilon > 0$, there must exist an index $N(\omega, \varepsilon)$ such that

$$
n \geq N(\omega, \varepsilon) \implies |X_n(\omega) - X(\omega)| < \varepsilon.
$$

This is exactly the deterministic $\varepsilon$-$N$ definition from the first section, applied to the numerical sequence $X_1(\omega), X_2(\omega), \ldots$ The index $N$ depends on $\omega$: different realisations may take longer to settle down. But for almost every $\omega$, settlement does eventually happen — the path enters the $\varepsilon$-band and never leaves.

### The Strong Law of Large Numbers

**Theorem (Strong Law of Large Numbers).** *Let $X_1, X_2, \ldots$ be i.i.d. real-valued random variables with $\mathbb{E}|X_1| < \infty$. Then*

$$
\bar X_n \to_{a.s.} \mathbb{E}(X_1).
$$

The SLLN says that with probability one, the realised running average enters every $\varepsilon$-band and remains there. Note that the SLLN requires only a finite first moment, whereas the WLLN (as stated above) assumes a finite variance — a stronger condition.

## 6. Convergence in probability vs almost surely

Both notions say that $X_n$ gets close to $X$, but they quantify "getting close" in fundamentally different ways. The cleanest way to see the distinction is through the language of failures.

Fix $\varepsilon > 0$ and say that a **failure** occurs at time $n$ if $|X_n(\omega) - X(\omega)| > \varepsilon$. Both convergence in probability and almost sure convergence imply that failures become rare, but they differ in how:

- **Convergence in probability:** for each fixed $n$, $\mathbb{P}(\text{failure at } n) \to 0$. At any single time, the proportion of paths outside the band is small.
- **Almost sure convergence:** for almost every $\omega$, there are only **finitely many** failures. Each individual path eventually stops leaving the band.

The first is a statement about vertical cross-sections — freeze time and ask how much mass lies outside the band. The second is a statement about horizontal trajectories — follow a single path and ask whether it ever leaves the band again.

### A counterexample: convergence in probability without almost sure convergence

Let $(X_n)$ be independent with $\mathbb{P}(X_n = 1) = 1/n$ and $\mathbb{P}(X_n = 0) = 1 - 1/n$. Since $\mathbb{P}(|X_n| > \varepsilon) = 1/n \to 0$ for any $\varepsilon \in (0, 1]$, we have $X_n \to_p 0$.

But does each path eventually stay at $0$? The probability of a spike at time $n$ is $1/n$, and $\sum_{n=1}^{\infty} 1/n = \infty$. By the second Borel–Cantelli lemma (which applies because the $X_n$ are independent), with probability $1$ infinitely many of the events $\{X_n = 1\}$ occur. Every path keeps producing spikes — they become sparser, but they never stop. So $X_n \not\to_{a.s.} 0$.

The key is the difference between "each individual failure becomes unlikely" and "the total number of failures is finite." Convergence in probability gives the first; almost sure convergence requires the second. The Borel–Cantelli lemma makes the gap precise: if the failure probabilities $p_n = \mathbb{P}(|X_n| > \varepsilon)$ satisfy $\sum p_n < \infty$, then almost sure convergence holds (first Borel–Cantelli). If $\sum p_n = \infty$ and the events are independent, then almost sure convergence fails (second Borel–Cantelli).

In [6]:
# Figure 8: almost sure convergence as eventual trapping of one path
n_max = 4000
eps = 0.08
while True:
    walk = RNG.choice([-1, 1], size=n_max)
    avg = np.cumsum(walk) / np.arange(1, n_max + 1)
    tail_max = np.maximum.accumulate(np.abs(avg[::-1]))[::-1]
    candidates = np.where(tail_max <= eps)[0]
    if len(candidates) > 0:
        N = int(candidates[0] + 1)
        break

fig, ax = plt.subplots(figsize=(9.3, 4.8))
ax.plot(np.arange(1, n_max + 1), avg, color="#1d4ed8", lw=1.2)
ax.axhspan(-eps, eps, color="#dbeafe", alpha=0.85)
ax.axhline(0.0, color="black", ls="--", lw=1.0)
ax.axvline(N, color="#dc2626", ls="--", lw=1.5)
ax.text(N + 70, eps + 0.012, rf"after $N={N}$ this path stays in the band", color="#991b1b")
ax.set_xlabel(r"time $n$")
ax.set_ylabel(r"running average $S_n/n$")
savefig("almost-sure-path.png")

content\blog\stochastic-convergence\almost-sure-path.png


WindowsPath('C:/Users/Alex/OneDrive - University of Cambridge/Desktop/Blog/alex-blog/content/blog/stochastic-convergence/almost-sure-path.png')

## 8. Working with convergence

Up to now, we have been asking what the different notions of convergence *mean*. In practice, though, asymptotic statistics is not built by proving every convergence from scratch. One establishes a small number of foundational results, the WLLN, the CLT, consistency of a particular estimator, and then uses a handful of general theorems to assemble more complex conclusions. This is where the subject starts to feel like ordinary analysis again: a toolkit of rules for passing limits through operations.

### Joint convergence

The first question is whether convergence of two sequences separately implies convergence of the pair. For convergence in probability and almost sure convergence, the answer is as clean as one could hope.

**Theorem (Joint convergence).** Let $X_n \in \mathbb{R}^d$ and $Y_n \in \mathbb{R}^k$. Then

$$
(X_n, Y_n) \to_p (X, Y) \iff X_n \to_p X \text{ and } Y_n \to_p Y,
$$

and the same equivalence holds with $\to_p$ replaced by $\to_{a.s.}$

Intuitively, controlling each coordinate separately is enough to control the whole vector, because closeness in $\mathbb{R}^{d+k}$ is just closeness in each coordinate simultaneously.

This matters constantly in practice. If one has separately established that $\bar X_n \to_p \mu$ and $\hat\sigma_n^2 \to_p \sigma^2$, the theorem immediately gives $(\bar X_n, \hat\sigma_n^2) \to_p (\mu, \sigma^2)$, putting both limits in a form that continuous functions can act on.

Convergence in distribution is more delicate. Knowing $X_n \to_d X$ and $Y_n \to_d Y$ separately says nothing about how the pair $(X_n, Y_n)$ behaves jointly, because distributional convergence controls marginals but not dependence structure. There is, however, one important special case: if one of the limits is a constant $c \in \mathbb{R}^k$, then

$$
(X_n, Y_n) \to_d (X, c) \iff X_n \to_d X \text{ and } Y_n \to_p c.
$$

This pattern, one sequence with a non-degenerate limiting distribution and another converging in probability to a fixed constant, is exactly the setting of Slutsky's lemma.

### The Continuous Mapping Theorem

Once one has a convergence in hand, the next question is whether applying a function to both sides preserves it. The answer is yes, as long as the function is continuous.

**Theorem (Continuous Mapping Theorem).** Let $g : \mathbb{R}^d \to \mathbb{R}^m$ be continuous at every point of a set $C$ with $\mathbb{P}(X \in C) = 1$. If $X_n \to X$ in any of the three modes (almost surely, in probability, or in distribution), then $g(X_n) \to g(X)$ in the same mode.

This is the stochastic analogue of the familiar fact that continuous functions preserve limits. For example, $X_n \to_p X$ implies $X_n^2 \to_p X^2$, $\exp(X_n) \to_p \exp(X)$, and $\sin(X_n) \to_p \sin(X)$; the same holds for convergence in distribution and almost sure convergence. Continuous transformations do not destroy asymptotic information: once a convergence is established, it propagates through any continuous operation.

### Slutsky's Lemma

Combining joint convergence and the continuous mapping theorem in the distributional setting gives what is perhaps the most-used result in asymptotic statistics.

**Theorem (Slutsky's Lemma).** Suppose $X_n \to_d X$ and $Y_n \to_p c$ for some deterministic constant $c$. Then

$$
X_n + Y_n \to_d X + c, \qquad Y_n X_n \to_d c X,
$$

and, if $c \neq 0$, $Y_n^{-1} X_n \to_d c^{-1} X$. The same holds when $Y_n$ is a random matrix converging in probability to a deterministic matrix $c$.

The logic is simple: because $Y_n \to_p c$, it behaves asymptotically as though it were already the constant $c$, so one may substitute $c$ for $Y_n$ inside any continuous expression in the limit. The canonical illustration is standardisation. Suppose the CLT gives $\sqrt{n}(\bar X_n - \mu) \to_d N(0, \sigma^2)$, and suppose $\hat\sigma_n \to_p \sigma$. Slutsky's lemma then gives

$$
\frac{\sqrt{n}(\bar X_n - \mu)}{\hat\sigma_n} \to_d N(0, 1).
$$

This mechanism underlies many asymptotic pivots in classical statistics.

### Passing to expectations: Dominated Convergence

The results so far concern the convergence of random variables themselves. A separate question is whether $W_n \to W$ implies $\mathbb{E}(W_n) \to \mathbb{E}(W)$. In general it does not: mass can drift into rare but large values, making the expectations diverge even as the random variables converge.

**Theorem (Dominated Convergence).** Suppose $W_n \to_p W$ and there exists an integrable random variable $V$ with $|W_n(\omega)| \leq |V(\omega)|$ for all $n$ and $\omega$. Then $\mathbb{E}|W| < \infty$ and

$$
\mathbb{E}(W_n) \to \mathbb{E}(W).
$$

The condition is that all terms in the sequence are dominated pointwise by a single integrable envelope. When this holds, the tails cannot escape to infinity, and expectation behaves continuously with respect to convergence. A representative application is in likelihood theory: to show continuity of $\theta \mapsto \mathbb{E}_{\theta_0} \log f(X, \theta)$, one shows pointwise convergence $\log f(X, \theta_n) \to \log f(X, \theta)$ and then uses dominated convergence to pass the limit through the expectation.

## 7. The hierarchy

The three notions form a strict implication chain:

$$
X_n \to_{a.s.} X \implies X_n \to_p X \implies X_n \to_d X.
$$

Neither reverse implication holds in general, as the counterexamples above demonstrate. The level of guarantee weakens as we move down: almost sure convergence controls paths, convergence in probability controls mass at each fixed time, and convergence in distribution controls only the law. A proof of both implications is given in Appendix A3.

## Appendix

The proofs below follow the presentation in Rajen Shah, *Principles of Statistics*, Sections 1.3–1.4.

### A1: Weak Law of Large Numbers

We use **Markov's inequality**: if $Z \geq 0$ then $\mathbb{P}(Z \geq t) \leq t^{-1}\mathbb{E}(Z)$.

*Proof.* Apply Markov's inequality to $(\bar X_n - \mu)^2$:

$$
\mathbb{P}(|\bar X_n - \mu| > \varepsilon) = \mathbb{P}\bigl((\bar X_n - \mu)^2 > \varepsilon^2\bigr) \leq \frac{\mathbb{E}(\bar X_n - \mu)^2}{\varepsilon^2} = \frac{\text{Var}(\bar X_n)}{\varepsilon^2} = \frac{\text{Var}(X_1)}{n\varepsilon^2} \to 0
$$

as $n \to \infty$. $\square$

### A2: Central Limit Theorem

The standard proof uses characteristic functions. We sketch the argument in the univariate case.

*Proof.* Write $Z_i = (X_i - \mu)/\sigma$ so that $\mathbb{E}(Z_i) = 0$ and $\text{Var}(Z_i) = 1$. The characteristic function of $Z_i$ satisfies

$$
\varphi_Z(t) = \mathbb{E}(e^{itZ_1}) = 1 - \frac{t^2}{2} + o(t^2)
$$

as $t \to 0$, using the expansion $e^{itZ_1} = 1 + itZ_1 - t^2Z_1^2/2 + \cdots$ and taking expectations. The quantity we want to study is $S_n = \frac{1}{\sqrt{n}}\sum_{i=1}^n Z_i$, whose characteristic function is

$$
\varphi_{S_n}(t) = \bigl[\varphi_Z(t/\sqrt{n})\bigr]^n = \Bigl[1 - \frac{t^2}{2n} + o(t^2/n)\Bigr]^n \to e^{-t^2/2}
$$

as $n \to \infty$. Since $e^{-t^2/2}$ is the characteristic function of $N(0,1)$, Lévy's continuity theorem gives $S_n \to_d N(0,1)$, which is equivalent to $\sqrt{n}(\bar X_n - \mu) \to_d N(0, \sigma^2)$. $\square$

### Sources

1. Rajen Shah, *Principles of Statistics*, Sections 1.3 and 1.4.
2. Robby McKilliam and others, ["Convergence in probability vs. almost sure convergence"](https://stats.stackexchange.com/questions/2230/convergence-in-probability-vs-almost-sure-convergence).
3. Pierre Lafaye de Micheaux and Benoit Liquet, ["Understanding Convergence Concepts: A Visual-Minded and Graphical Simulation-Based Approach"](https://doi.org/10.1198/tas.2009.0032).

In [7]:
# Figure 9: hierarchy diagram
fig, ax = plt.subplots(figsize=(8, 6))
ax.axis("off")

boxes = [
    (0.5, 0.82, "Almost sure convergence", "paths settle\nSLLN"),
    (0.5, 0.50, "Convergence in probability", "mass settles\nWLLN"),
    (0.5, 0.18, "Convergence in distribution", "laws settle\nCLT"),
]
colors = ["#dbeafe", "#fde68a", "#fecaca"]

for (x0, y0, title, subtitle), color in zip(boxes, colors):
    ax.text(
        x0,
        y0,
        f"{title}\n{subtitle}",
        ha="center",
        va="center",
        fontsize=14,
        bbox={"boxstyle": "round,pad=0.6", "facecolor": color, "edgecolor": "#334155", "linewidth": 1.5},
    )

ax.annotate("", xy=(0.5, 0.62), xytext=(0.5, 0.70), arrowprops={"arrowstyle": "->", "lw": 2, "color": "#334155"})
ax.annotate("", xy=(0.5, 0.30), xytext=(0.5, 0.38), arrowprops={"arrowstyle": "->", "lw": 2, "color": "#334155"})
ax.text(0.56, 0.66, "implies", fontsize=12, color="#334155")
ax.text(0.56, 0.34, "implies", fontsize=12, color="#334155")
savefig("hierarchy.png")

print("Notebook figures are ready in", OUTDIR)

content\blog\stochastic-convergence\hierarchy.png
Notebook figures are ready in C:\Users\Alex\OneDrive - University of Cambridge\Desktop\Blog\alex-blog\content\blog\stochastic-convergence
